# Clean QAChat Repository

This auxiliary notebook removes common generated cache files and
operating-system metadata from the QAChat repository.

It is intended to live in the `assets/` directory. The repository root is
located automatically from the `qachat` folder, so the notebook also works
when launched from another directory.


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def find_repository_root(start: Path | None = None) -> Path:
    """
    Find the QAChat repository root.

    Parameters
    ----------
    start : Path | None
        Directory from which to begin the search.

    Returns
    -------
    Path
        Absolute path to the QAChat repository root.

    Raises
    ------
    RuntimeError
        If the repository root cannot be found.

    """

    current = (start or Path.cwd()).resolve()

    for directory in (current, *current.parents):
        if (directory / "qachat").is_dir():
            return directory

    raise RuntimeError(
        "Could not locate the QAChat repository root "
        f"when starting from {current}."
    )


In [ ]:
repository_root = find_repository_root()

print(f"Repository root: {repository_root}")


## Cleanup rules

The rules below target generated cache files, notebook checkpoints,
coverage artifacts, and operating-system metadata. Source files,
project data, Git metadata, and virtual environments are left alone.


In [ ]:
DIRECTORY_NAMES = {
    "__pycache__",
    "__MACOSX",
    ".ipynb_checkpoints",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".coverage_cache",
}

FILE_NAMES = {
    ".DS_Store",
    ".coverage",
    "Thumbs.db",
}

FILE_SUFFIXES = {
    ".pyc",
    ".pyo",
}

SKIP_DIRECTORY_NAMES = {
    ".git",
    ".venv",
    "venv",
    "env",
}


## Find temporary paths

This cell collects all matching paths before anything is deleted. It
does not descend into Git metadata or common virtual-environment
directories.


In [ ]:
def is_skipped(path: Path, root: Path) -> bool:
    """Return whether a path lies inside a skipped directory."""

    relative_parts = path.relative_to(root).parts
    return any(
        part in SKIP_DIRECTORY_NAMES for part in relative_parts
    )


In [ ]:
def find_temporary_paths(
    root: Path,
) -> tuple[list[Path], list[Path]]:
    """Find temporary directories and files below a root directory.

    Parameters
    ----------
    root : Path
        Root directory to inspect.

    Returns
    -------
    tuple[list[Path], list[Path]]
        Temporary directories and files, respectively.

    """

    directories: list[Path] = []
    files: list[Path] = []

    for path in root.rglob("*"):
        if is_skipped(path, root):
            continue

        if path.is_dir() and path.name in DIRECTORY_NAMES:
            directories.append(path)
        elif path.is_file() and (
            path.name in FILE_NAMES
            or path.suffix in FILE_SUFFIXES
        ):
            files.append(path)

    directories.sort(
        key=lambda path: len(path.parts),
        reverse=True,
    )
    files.sort()

    return directories, files


In [ ]:
directories, files = find_temporary_paths(repository_root)


In [ ]:
print("Directories")
print("-----------")
for path in directories:
    print(path.relative_to(repository_root))

print("\nSummary")
print("-------")
print(f"{len(directories)} directories")


In [ ]:
print("\nFiles")
print("-----")
for path in files:
    print(path.relative_to(repository_root))

print("\nSummary")
print("-------")
print(f"{len(files)} files")


## Remove temporary paths


In [ ]:
DELETE = True  # Set `DELETE = False` to run a preview.


In [ ]:
if not DELETE:
    print("Dry run only. Set DELETE = True to remove the paths.")
else:
    removed_directories = 0
    removed_files = 0

    for path in files:
        if path.exists():
            path.unlink()
            removed_files += 1

    for path in directories:
        if path.exists():
            shutil.rmtree(path)
            removed_directories += 1

    print(
        f"Removed {removed_directories} directories and {removed_files} files."
    )


## Verify cleanup

Run this final cell after deletion to confirm that no matching
temporary paths remain outside the skipped directories.


In [ ]:
remaining_directories, remaining_files = find_temporary_paths(
    repository_root
)

if remaining_directories or remaining_files:
    print("Temporary paths remain:")
    for path in remaining_directories + remaining_files:
        print(path.relative_to(repository_root))
else:
    print("Repository is clean.")
